In [1]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

# 1. Semente para reprodutibilidade dos experimentos
SEED = 42
np.random.seed(SEED)

print("Gerando matriz fiscal sintética...")

# 2. Geração de 10.000 instâncias fiscais (7 informativas e 3 de ruído puro)
#    - n_redundant=0 e n_repeated=0: nenhuma coluna é combinação das outras
#    - n_features - n_informative = 3 colunas sobram e viram ruído gaussiano puro
#    - shuffle=False: mantém a ordem das colunas (1 a 7 informativas, 8 a 10 ruído),
#      o que torna a verdade fundamental (ground truth) conhecida
X, y = make_classification(
    n_samples=10000,
    n_features=10,
    n_informative=7,
    n_redundant=0,
    n_repeated=0,
    n_classes=2,
    weights=[0.85, 0.15],  # 85% regulares e 15% suspeitos
    shuffle=False,
    random_state=SEED,
)

# 3. Mapeamento das colunas para o domínio tributário
colunas_informativas = [
    "faturamento_anual",
    "imposto_declarado",
    "margem_lucro_presumido",
    "volume_notas_emitidas",
    "deducoes_fiscais",
    "movimentacao_bancaria",
    "creditos_tributarios",
]
colunas_ruido = ["ruido_mercado_1", "ruido_mercado_2", "ruido_mercado_3"]
colunas_fiscais = colunas_informativas + colunas_ruido

# 4. Estruturação do DataFrame
df_experimento = pd.DataFrame(X, columns=colunas_fiscais)
df_experimento["risco_fiscal"] = y

# 5. Embaralha apenas as LINHAS (as colunas mantêm a ordem), pois shuffle=False
#    deixa as instâncias agrupadas por classe
df_experimento = df_experimento.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 6. Verificações de sanidade
print(f"Dimensões: {df_experimento.shape}")
print("Proporção das classes:")
print(df_experimento["risco_fiscal"].value_counts(normalize=True).round(3))
posto = np.linalg.matrix_rank(df_experimento[colunas_fiscais].values)
print(f"Posto da matriz de variáveis: {posto} (esperado: 10, sem colunas redundantes)")

# 7. Exportação do dataset e dos metadados (ground truth para o XAI)
os.makedirs("../dados", exist_ok=True)
df_experimento.to_csv("../dados/dataset_fiscal.csv", index=False)

with open("../dados/metadados.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "semente": SEED,
            "colunas_informativas": colunas_informativas,
            "colunas_ruido": colunas_ruido,
            "alvo": "risco_fiscal",
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Dataset gerado e salvo com sucesso em: dados/dataset_fiscal.csv")

Gerando matriz fiscal sintética...
Dimensões: (10000, 11)
Proporção das classes:
risco_fiscal
0    0.846
1    0.154
Name: proportion, dtype: float64
Posto da matriz de variáveis: 10 (esperado: 10, sem colunas redundantes)
Dataset gerado e salvo com sucesso em: dados/dataset_fiscal.csv
